In [ ]:
import datetime, os, logging, pandas as pd, requests
from garminconnect import Garmin
logging.getLogger("garth.http").setLevel(logging.CRITICAL)
logging.getLogger("garth").setLevel(logging.CRITICAL)

def get_yesterday_garmin_data():
    EMAIL = "letungquan97@gmail.com"
    PASSWORD = "Workforce@210997"
    TOKEN_DIR = "garmin_tokens"
    WEBHOOK_URL = "https://n8n.pixelterminal.com/webhook/35cd3382-dab9-4a56-b540-9e43bcd0dee2"
    yesterday_date = datetime.date.today() - datetime.timedelta(days=1)
    yesterday_iso = yesterday_date.isoformat()
    yesterday_str = yesterday_date.strftime("%m/%d/%Y") 
    print(f"Đang phân tích dữ liệu Body Composition cho ngày: {yesterday_str}...")
    try:
        client = Garmin(EMAIL, PASSWORD)
        login_success = False       
        try:
            client.login(TOKEN_DIR)
            print("Đăng nhập siêu tốc bằng Token thành công! (Bypass Garmin Firewall)")
            login_success = True
        except Exception:
            print("Chưa có Token hợp lệ, đang tiến hành đăng nhập bằng mật khẩu...")
        if not login_success:
            client.login() 
            print("Đăng nhập bằng mật khẩu thành công!")
            if not os.path.exists(TOKEN_DIR):
                os.makedirs(TOKEN_DIR)
            try:
                client.garth.dump(TOKEN_DIR)
            except AttributeError:
                import garth
                garth.client.dump(TOKEN_DIR)
            print(f"Đã lưu Session Token an toàn vào thư mục '{TOKEN_DIR}'.")
        # Kéo dữ liệu
        body_comp = client.get_body_composition(yesterday_iso)
        weight, body_fat, bmi, muscle_mass, bone_mass, body_water = "", "", "", "", "", ""
        if body_comp and 'dateWeightList' in body_comp and len(body_comp['dateWeightList']) > 0:
            data = body_comp['dateWeightList'][0]            
            def grams_to_kg(val):
                if val is not None and str(val).strip() != "":
                    return round(float(val) / 1000, 2)
                return ""    
            weight = grams_to_kg(data.get('weight'))
            muscle_mass = grams_to_kg(data.get('muscleMass'))
            bone_mass = grams_to_kg(data.get('boneMass'))
            body_fat = data.get('bodyFat', "")
            bmi = data.get('bmi', "")
            body_water = data.get('bodyWater', "")     
        # 1. ĐÓNG GÓI DỮ LIỆU ĐỂ BẮN LÊN n8n
        payload = {
            "date": yesterday_str,
            "weight": weight,
            "bodyfat": body_fat,
            "bmi": bmi,
            "skeletal_muscle_mass": muscle_mass,
            "bone_mass": bone_mass,
            "body_water": body_water
        }
        # 2. THỰC THI GỬI WEBHOOK (Có bọc lót chống sập mạng)
        print("\nĐang thiết lập liên kết n8n...")
        try:
            # Gửi method POST kèm JSON, timeout 10 giây
            response = requests.post(WEBHOOK_URL, json=payload, timeout=10)
            
            # Kiểm tra trạng thái trả về từ server n8n của anh
            if response.status_code == 200:
                print(">>> [THÀNH CÔNG] Dữ liệu đã được bắn trúng đích vào n8n Webhook!")
            else:
                print(f">>> [CẢNH BÁO] n8n phản hồi mã lỗi: {response.status_code}")
                print(f"Chi tiết: {response.text}")
        except requests.exceptions.RequestException as req_err:
            print(f">>> [THẤT BẠI] Không thể kết nối tới máy chủ n8n. Lỗi: {req_err}")
        # 3. VẪN LƯU BẢN SAO EXCEL DỰ PHÒNG CHỐNG MẤT MÁT (Data Backup)
        df = pd.DataFrame([payload])
        # Format lại tên cột cho giống template Excel cũ
        df.columns = ['date', 'weight', 'bodyfat', 'BMI', 'skeletal muscle mass', 'bone mass', 'body water']
        output_file = f"Garmin_Data_{yesterday_iso}.xlsx"
        df.to_excel(output_file, index=False)  
        print(f"\nTuyệt vời! Bản sao lưu Local Excel đã tạo tại: {output_file}")
    except Exception as e:
        print(f"\nOái, hệ thống báo lỗi không lường trước được: {e}")

if __name__ == "__main__":
    get_yesterday_garmin_data()